# MLP

simplest model - flatten 48x48 and dense layers. no CNN, no augmentation.

In [3]:
import sys
sys.path.append(".")

from src.data import make_loaders

loaders = make_loaders(augment=False)
print({k: len(v.dataset) for k, v in loaders.items()})

{'train': 28709, 'val': 3589, 'test': 3589}


## sanity check

forward + backward erts batch-ze, sanam vtsvrtni. loss ~ ln(7)=1.95 unda iyos da gradientebi unda chamodiodes.

In [5]:
#sanity check forward + backward on one batch
import torch
from src.models import build_model
from src.engine import get_device

device = get_device()
model = build_model("mlp", hidden=512).to(device)
x, y = next(iter(loaders["train"]))
x, y = x.to(device), y.to(device)

out = model(x)
print("forward:", tuple(out.shape), "init loss:", round(torch.nn.functional.cross_entropy(out, y).item(), 3))

loss = torch.nn.functional.cross_entropy(out, y)
loss.backward()
grads_ok = all(p.grad is not None and p.grad.abs().sum() > 0 for p in model.parameters())
print("backward: grads flow =", bool(grads_ok))

forward: (128, 7) init loss: 1.93
backward: grads flow = True


In [4]:
from src import notebook_utils as nu

nu.setup()

grid = [
    {"note": "underfit", "model": {"hidden": 64,   "dropout": 0.5}, "train": {"epochs": 20, "lr": 1e-3}},
    {"note": "wellfit_a", "model": {"hidden": 512,  "dropout": 0.3}, "train": {"epochs": 20, "lr": 1e-3}},
    {"note": "wellfit_b", "model": {"hidden": 512,  "dropout": 0.5}, "train": {"epochs": 20, "lr": 5e-4}},
    {"note": "overfit",  "model": {"hidden": 1024, "dropout": 0.0}, "train": {"epochs": 20, "lr": 1e-3}},
    {"note": "lr_high", "model": {"hidden": 512, "dropout": 0.3}, "train": {"epochs": 20, "lr": 1e-2}},
]

nu.run_architecture(
    "MLP",
    grid=grid,
    final_train={"epochs": 40},
    plot=("MLP", "mlp_curves.png"),
    augment=False,
)

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: sekhv23 (sekhv23-free-university-of-tbilisi) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


MLP: output (128, 7), loss=1.938


wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


epoch   1  train_acc 0.2554  val_acc 0.3285  train_loss 1.8072  val_loss 1.7007
epoch   2  train_acc 0.3118  val_acc 0.3650  train_loss 1.7245  val_loss 1.6542
epoch   3  train_acc 0.3255  val_acc 0.3583  train_loss 1.6979  val_loss 1.6342
epoch   4  train_acc 0.3372  val_acc 0.3636  train_loss 1.6719  val_loss 1.6062
epoch   5  train_acc 0.3413  val_acc 0.3725  train_loss 1.6627  val_loss 1.6036
epoch   6  train_acc 0.3471  val_acc 0.3767  train_loss 1.6502  val_loss 1.5942
epoch   7  train_acc 0.3537  val_acc 0.3823  train_loss 1.6374  val_loss 1.5857
epoch   8  train_acc 0.3544  val_acc 0.3792  train_loss 1.6332  val_loss 1.5756
epoch   9  train_acc 0.3582  val_acc 0.3801  train_loss 1.6227  val_loss 1.5721
epoch  10  train_acc 0.3604  val_acc 0.3859  train_loss 1.6160  val_loss 1.5601
epoch  11  train_acc 0.3646  val_acc 0.3879  train_loss 1.6098  val_loss 1.5622
epoch  12  train_acc 0.3667  val_acc 0.3881  train_loss 1.6034  val_loss 1.5600
epoch  13  train_acc 0.3693  val_acc 0.3

lr,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_acc,▁▄▅▅▆▆▆▆▇▇▇▇▇▇██████
train_loss,█▆▅▄▄▃▃▃▃▂▂▂▂▂▁▁▁▁▁▁
val_acc,▁▅▄▄▅▆▆▆▆▇▇▇▇▇▆██▇██
val_loss,█▆▅▄▄▃▃▃▃▂▂▂▂▂▂▁▁▁▁▁
best_epoch,17
best_val_acc,0.39955
final_train_acc,0.38271
lr,0.001
n_params,149831
overfit_gap,-0.02229


...

     note           run  val_acc     gap  score
wellfit_b MLP_wellfit_b   0.4536  0.0421 0.4536
wellfit_a MLP_wellfit_a   0.4642  0.0766 0.4509
 underfit  MLP_underfit   0.3996 -0.0223 0.3996
  overfit   MLP_overfit   0.4667  0.4362 0.2736
  lr_high   MLP_lr_high   0.2499 -0.0107 0.2499

best: wellfit_b  val_acc=0.4536  gap=0.0421


epoch   1  train_acc 0.3017  val_acc 0.3720  train_loss 1.7373  val_loss 1.6335
epoch   2  train_acc 0.3519  val_acc 0.3920  train_loss 1.6523  val_loss 1.5959
epoch   3  train_acc 0.3708  val_acc 0.3906  train_loss 1.6148  val_loss 1.5723
epoch   4  train_acc 0.3789  val_acc 0.4009  train_loss 1.5852  val_loss 1.5498
epoch   5  train_acc 0.3943  val_acc 0.4079  train_loss 1.5576  val_loss 1.5379
epoch   6  train_acc 0.4040  val_acc 0.4202  train_loss 1.5314  val_loss 1.5238
epoch   7  train_acc 0.4111  val_acc 0.4171  train_loss 1.5105  val_loss 1.5050
epoch   8  train_acc 0.4216  val_acc 0.4230  train_loss 1.4907  val_loss 1.5013
epoch   9  train_acc 0.4252  val_acc 0.4319  train_loss 1.4754  val_loss 1.4856
epoch  10  train_acc 0.4359  val_acc 0.4283  train_loss 1.4573  val_loss 1.4778
epoch  11  train_acc 0.4402  val_acc 0.4269  train_loss 1.4390  val_loss 1.4749
epoch  12  train_acc 0.4475  val_acc 0.4271  train_loss 1.4225  val_loss 1.4729
epoch  13  train_acc 0.4554  val_acc 0.4

lr,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_acc,▁▂▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇██████
train_loss,█▇▇▆▆▆▆▅▅▅▅▅▅▄▄▄▄▄▄▃▃▃▃▃▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁
val_acc,▁▂▂▃▄▄▄▅▅▅▅▅▅▅▅▆▆▆▅▆▆▇▇▇▇▇▇▇▇▇▇█▇▇▇██▇██
val_loss,█▇▆▅▅▄▄▃▃▃▃▃▂▂▂▂▂▂▂▁▁▁▂▁▁▁▁▁▁▂▁▂▂▂▂▂▂▂▂▂
best_epoch,37
best_val_acc,0.472
final_train_acc,0.59152
lr,0.0005
n_params,1313287
overfit_gap,0.10832


{'best_val_acc': 0.47199777096684314, 'overfit_gap': 0.10832449465815686, 'selection_score': 0.44283552363776474, 'best_epoch': 37}


(        note            run  val_acc     gap   score
 2  wellfit_b  MLP_wellfit_b   0.4536  0.0421  0.4536
 1  wellfit_a  MLP_wellfit_a   0.4642  0.0766  0.4509
 0   underfit   MLP_underfit   0.3996 -0.0223  0.3996
 3    overfit    MLP_overfit   0.4667  0.4362  0.2736
 4    lr_high    MLP_lr_high   0.2499 -0.0107  0.2499,
 {'note': 'wellfit_b',
  'model': {'hidden': 512, 'dropout': 0.5},
  'train': {'epochs': 20, 'lr': 0.0005}},
 {'best_val_acc': 0.4536082474226804,
  'overfit_gap': 0.04213673025589104,
  'selection_score': 0.4536082474226804,
  'best_epoch': 20})